# Notebook 5 — Personalized Recommendation Engine

**Algorithm 2:** Given a customer persona and context (time, weather, budget), recommend the best drink + customization to maximize satisfaction and revenue.

**Approach:** Content-based filtering using product attributes (calories, sugar, caffeine, price) matched against persona constraints. No collaborative filtering — since our purchase data is synthetic, recommendations must be grounded in product attributes (real data), not user-item interactions (synthetic).

**Output:** For any persona + context → Top-3 recommendations with customization suggestions and expected ticket price.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display, HTML

np.random.seed(42)

ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    DATA_DIR = Path('/kaggle/input/starbucks-recommendation-engine')
else:
    DATA_DIR = Path('../data')

menu = pd.read_csv((DATA_DIR / 'raw' if not ON_KAGGLE else DATA_DIR) / 'starbucks_menu.csv')
txn = pd.read_csv((DATA_DIR / 'processed' if not ON_KAGGLE else DATA_DIR) / 'synthetic_transactions.csv')
fred = pd.read_csv((DATA_DIR / 'raw' if not ON_KAGGLE else DATA_DIR) / 'fred_macro.csv', parse_dates=['date'])

print(f'Menu:         {menu.shape}')
print(f'Transactions: {txn.shape}')

## Section 1 — Persona Profiles & Constraints

Each persona has explicit constraints that the recommendation engine must satisfy.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PERSONA CONSTRAINT PROFILES
# ══════════════════════════════════════════════════════════════════════

PERSONA_CONSTRAINTS = {
    'morning_commuter': {
        'description': 'Needs caffeine, quick, moderate budget',
        'max_calories': 400,
        'min_caffeine_mg': 75,
        'max_price': 6.00,
        'preferred_categories': ['Coffee', 'Classic Espresso Drinks'],
        'time_slots': ['early_morning', 'morning_rush'],
        'customization_budget': 1.50,
    },
    'afternoon_treat': {
        'description': 'Indulgent, willing to spend, flavor-focused',
        'max_calories': 600,
        'min_caffeine_mg': 0,
        'max_price': 7.50,
        'preferred_categories': ['Frappuccino® Blended Coffee', 'Frappuccino® Blended Crème',
                                  'Signature Espresso Drinks'],
        'time_slots': ['lunch', 'afternoon'],
        'customization_budget': 2.50,
    },
    'health_conscious': {
        'description': 'Low calorie, low sugar, willing to pay for quality',
        'max_calories': 200,
        'min_caffeine_mg': 0,
        'max_price': 6.50,
        'preferred_categories': ['Coffee', 'Tazo® Tea Drinks', 'Frappuccino® Light Blended Coffee'],
        'time_slots': ['morning_rush', 'late_morning'],
        'customization_budget': 1.00,
    },
    'student': {
        'description': 'Budget-conscious, needs energy, likes sweet drinks',
        'max_calories': 500,
        'min_caffeine_mg': 50,
        'max_price': 5.50,
        'preferred_categories': ['Frappuccino® Blended Coffee', 'Shaken Iced Beverages',
                                  'Classic Espresso Drinks'],
        'time_slots': ['late_morning', 'afternoon', 'evening'],
        'customization_budget': 1.00,
    },
    'weekend_explorer': {
        'description': 'Adventurous, tries new things, moderate-high budget',
        'max_calories': 500,
        'min_caffeine_mg': 0,
        'max_price': 7.00,
        'preferred_categories': ['Signature Espresso Drinks', 'Smoothies',
                                  'Frappuccino® Blended Coffee'],
        'time_slots': ['late_morning', 'lunch', 'afternoon'],
        'customization_budget': 2.00,
    },
}

# Display as table
constraint_rows = []
for name, c in PERSONA_CONSTRAINTS.items():
    constraint_rows.append({
        'Persona': name,
        'Description': c['description'],
        'Max Cal': c['max_calories'],
        'Min Caffeine': c['min_caffeine_mg'],
        'Max Price': f"${c['max_price']:.2f}",
        'Custom Budget': f"${c['customization_budget']:.2f}",
    })
display(pd.DataFrame(constraint_rows))

## Section 2 — Recommendation Engine

The engine scores every menu item against a persona's constraints and context, then returns the top-N with customization suggestions.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RECOMMENDATION ENGINE
# ══════════════════════════════════════════════════════════════════════

CUSTOMIZATIONS = [
    {'name': 'Extra Shot',      'price': 0.80, 'calories': 5,  'caffeine': 75, 'sugar': 0},
    {'name': 'Vanilla Syrup',   'price': 0.60, 'calories': 20, 'caffeine': 0,  'sugar': 5},
    {'name': 'Caramel Drizzle', 'price': 0.60, 'calories': 15, 'caffeine': 0,  'sugar': 2},
    {'name': 'Oat Milk',        'price': 0.70, 'calories': 10, 'caffeine': 0,  'sugar': 1},
    {'name': 'Cold Foam',       'price': 1.00, 'calories': 35, 'caffeine': 0,  'sugar': 3},
    {'name': 'Sugar-Free Vanilla', 'price': 0.60, 'calories': 0, 'caffeine': 0, 'sugar': -5},
    {'name': 'Blonde Espresso', 'price': 0.00, 'calories': 0,  'caffeine': 10, 'sugar': 0},
]


def score_item(item, persona_name, temp_f, hour):
    """Score a single menu item for a persona in context."""
    constraints = PERSONA_CONSTRAINTS[persona_name]
    
    score = 0.0
    reasons = []
    
    # --- Hard constraints (pass/fail) ---
    if item['calories'] > constraints['max_calories']:
        return -1, ['Exceeds calorie limit']
    if item['caffeine_mg'] < constraints['min_caffeine_mg']:
        return -1, ['Below caffeine minimum']
    if item['price_usd'] > constraints['max_price']:
        return -1, ['Exceeds price limit']
    
    # --- Category preference (0-30 pts) ---
    if item['category'] in constraints['preferred_categories']:
        score += 30
        reasons.append('Preferred category')
    else:
        score += 10
    
    # --- Temperature fit (0-20 pts) ---
    is_cold_drink = item['category'] in [
        'Frappuccino® Blended Coffee', 'Frappuccino® Blended Crème',
        'Frappuccino® Light Blended Coffee', 'Shaken Iced Beverages', 'Smoothies'
    ]
    if temp_f > 75 and is_cold_drink:
        score += 20
        reasons.append('Cold drink for hot weather')
    elif temp_f < 55 and not is_cold_drink:
        score += 20
        reasons.append('Hot drink for cold weather')
    else:
        score += 10
    
    # --- Time fit (0-15 pts) ---
    time_slot = 'morning_rush' if 7 <= hour < 10 else 'late_morning' if 10 <= hour < 12 else \
                'lunch' if 12 <= hour < 14 else 'afternoon' if 14 <= hour < 17 else \
                'evening' if 17 <= hour < 21 else 'early_morning'
    if time_slot in constraints['time_slots']:
        score += 15
        reasons.append('Good time slot fit')
    else:
        score += 5
    
    # --- Value score (0-20 pts) ---
    # Higher calories per dollar = better value (for indulgent personas)
    if persona_name in ['afternoon_treat', 'student']:
        cal_per_dollar = item['calories'] / max(item['price_usd'], 0.01)
        score += min(20, cal_per_dollar / 5)
    else:
        # For health-conscious: lower calories = better
        score += max(0, 20 - item['calories'] / 25)
    
    # --- Seasonal bonus (0-15 pts) ---
    if item['seasonal_flag'] == 1:
        score += 15
        reasons.append('Seasonal special')
    
    return round(score, 2), reasons


def suggest_customizations(item, persona_name, temp_f):
    """Suggest customizations that fit within persona's budget and constraints."""
    constraints = PERSONA_CONSTRAINTS[persona_name]
    budget = constraints['customization_budget']
    max_cal = constraints['max_calories'] - item['calories']
    
    suggestions = []
    remaining_budget = budget
    
    for cust in CUSTOMIZATIONS:
        if cust['price'] > remaining_budget:
            continue
        if item['calories'] + cust['calories'] > constraints['max_calories']:
            continue
        
        # Persona-specific logic
        benefit = 0
        reason = ''
        
        if persona_name == 'morning_commuter' and cust['name'] == 'Extra Shot':
            benefit = 5; reason = 'Extra caffeine for focus'
        elif persona_name == 'health_conscious' and cust['name'] == 'Sugar-Free Vanilla':
            benefit = 5; reason = 'Flavor without sugar'
        elif persona_name == 'health_conscious' and cust['name'] == 'Oat Milk':
            benefit = 4; reason = 'Plant-based alternative'
        elif persona_name == 'afternoon_treat' and cust['name'] in ['Caramel Drizzle', 'Cold Foam']:
            benefit = 4; reason = 'Extra indulgence'
        elif persona_name == 'student' and cust['name'] == 'Extra Shot':
            benefit = 4; reason = 'Study fuel'
        elif persona_name == 'weekend_explorer' and cust['name'] in ['Oat Milk', 'Cold Foam']:
            benefit = 4; reason = 'Try something new'
        elif temp_f > 75 and cust['name'] == 'Cold Foam':
            benefit = 3; reason = 'Refreshing addition'
        elif cust['name'] == 'Blonde Espresso' and cust['price'] == 0:
            benefit = 2; reason = 'Free flavor upgrade'
        
        if benefit > 0:
            suggestions.append({
                'name': cust['name'],
                'price': cust['price'],
                'benefit': benefit,
                'reason': reason,
            })
            remaining_budget -= cust['price']
    
    return sorted(suggestions, key=lambda x: -x['benefit'])[:3]


def recommend(persona_name, temp_f=70, hour=9, top_n=3):
    """Full recommendation: score all items, return top-N with customizations."""
    scored = []
    for _, item in menu.iterrows():
        score, reasons = score_item(item, persona_name, temp_f, hour)
        if score > 0:
            custs = suggest_customizations(item, persona_name, temp_f)
            total_upcharge = sum(c['price'] for c in custs)
            scored.append({
                'product_name': item['product_name'],
                'category': item['category'],
                'size': item['size'],
                'base_price': item['price_usd'],
                'calories': item['calories'],
                'sugar_g': item['sugar_g'],
                'caffeine_mg': item['caffeine_mg'],
                'score': score,
                'reasons': reasons,
                'customizations': custs,
                'total_price': round(item['price_usd'] + total_upcharge, 2),
            })
    
    # Sort by score, deduplicate by product name (keep best size)
    scored.sort(key=lambda x: -x['score'])
    seen = set()
    unique = []
    for s in scored:
        if s['product_name'] not in seen:
            seen.add(s['product_name'])
            unique.append(s)
    
    return unique[:top_n]


print('Recommendation engine ready.')

## Section 3 — Demo: Recommendations for All Personas

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DEMO: ALL PERSONAS
# ══════════════════════════════════════════════════════════════════════

demo_scenarios = [
    {'persona': 'morning_commuter', 'temp_f': 45, 'hour': 8,
     'context': 'Weekday morning, 45°F, needs caffeine'},
    {'persona': 'afternoon_treat', 'temp_f': 82, 'hour': 14,
     'context': 'Summer afternoon, 82°F, wants indulgence'},
    {'persona': 'health_conscious', 'temp_f': 70, 'hour': 10,
     'context': 'Spring morning, 70°F, watching calories'},
    {'persona': 'student', 'temp_f': 60, 'hour': 16,
     'context': 'Study session, 60°F, tight budget'},
    {'persona': 'weekend_explorer', 'temp_f': 75, 'hour': 11,
     'context': 'Saturday brunch, 75°F, feeling adventurous'},
]

for scenario in demo_scenarios:
    print('\n' + '=' * 70)
    print(f"PERSONA: {scenario['persona'].upper()}")
    print(f"Context: {scenario['context']}")
    print('=' * 70)
    
    recs = recommend(scenario['persona'], scenario['temp_f'], scenario['hour'])
    
    for rank, rec in enumerate(recs, 1):
        print(f"\n  #{rank} {rec['product_name']} ({rec['size']})")
        print(f"     Category:  {rec['category']}")
        print(f"     Calories:  {rec['calories']} kcal | Sugar: {rec['sugar_g']}g | Caffeine: {rec['caffeine_mg']}mg")
        print(f"     Base:      ${rec['base_price']:.2f}")
        print(f"     Score:     {rec['score']}")
        
        if rec['customizations']:
            print(f"     Customizations:")
            for c in rec['customizations']:
                print(f"       + {c['name']} (+${c['price']:.2f}) — {c['reason']}")
            print(f"     Total:     ${rec['total_price']:.2f}")
        
        if rec['reasons']:
            print(f"     Why:       {', '.join(rec['reasons'])}")

## Section 4 — Revenue Impact: Customization Upsell Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# UPSELL REVENUE ANALYSIS
# ══════════════════════════════════════════════════════════════════════

# Run recommendations for all personas across multiple conditions
conditions = [
    (40, 8), (55, 8), (70, 8), (85, 8),     # morning, varying temp
    (40, 14), (55, 14), (70, 14), (85, 14),  # afternoon, varying temp
]

upsell_data = []
for persona in PERSONA_CONSTRAINTS:
    for temp, hour in conditions:
        recs = recommend(persona, temp, hour, top_n=1)
        if recs:
            rec = recs[0]
            upsell = rec['total_price'] - rec['base_price']
            upsell_data.append({
                'persona': persona,
                'temp_f': temp,
                'hour': hour,
                'base_price': rec['base_price'],
                'total_price': rec['total_price'],
                'upsell': upsell,
                'n_customizations': len(rec['customizations']),
            })

upsell_df = pd.DataFrame(upsell_data)

# Summary
print('=== Upsell Revenue by Persona ===')
persona_upsell = upsell_df.groupby('persona').agg(
    avg_base=('base_price', 'mean'),
    avg_total=('total_price', 'mean'),
    avg_upsell=('upsell', 'mean'),
    avg_customizations=('n_customizations', 'mean'),
).round(2)
persona_upsell['upsell_pct'] = (persona_upsell['avg_upsell'] / persona_upsell['avg_base'] * 100).round(1)
display(persona_upsell)

overall_upsell_pct = upsell_df['upsell'].sum() / upsell_df['base_price'].sum() * 100
print(f'\nOverall upsell: {overall_upsell_pct:.1f}% revenue lift from customization recommendations')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].barh(persona_upsell.index, persona_upsell['avg_upsell'], color='#00704A', edgecolor='white')
axes[0].set_title('Average Upsell per Recommendation ($)')
axes[0].set_xlabel('Upsell ($)')
for i, v in enumerate(persona_upsell['avg_upsell']):
    axes[0].text(v + 0.02, i, f'${v:.2f}', va='center', fontsize=9)

axes[1].barh(persona_upsell.index, persona_upsell['upsell_pct'], color='#E76F51', edgecolor='white')
axes[1].set_title('Upsell as % of Base Price')
axes[1].set_xlabel('Upsell %')
for i, v in enumerate(persona_upsell['upsell_pct']):
    axes[1].text(v + 0.2, i, f'{v:.1f}%', va='center', fontsize=9)

plt.suptitle('Revenue Impact of Personalized Customization Recommendations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 5 — Interactive Scenario Explorer

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SCENARIO MATRIX: temp × persona heatmap of recommended prices
# ══════════════════════════════════════════════════════════════════════

temps = [30, 40, 50, 60, 70, 80, 90]
personas = list(PERSONA_CONSTRAINTS.keys())

price_matrix = np.zeros((len(personas), len(temps)))
for i, persona in enumerate(personas):
    for j, temp in enumerate(temps):
        recs = recommend(persona, temp, 12, top_n=1)
        if recs:
            price_matrix[i, j] = recs[0]['total_price']

fig_heat = go.Figure(data=go.Heatmap(
    z=price_matrix,
    x=[f'{t}°F' for t in temps],
    y=personas,
    text=[[f'${v:.2f}' for v in row] for row in price_matrix],
    texttemplate='%{text}',
    colorscale='Greens',
    colorbar_title='Total Price ($)',
))

fig_heat.update_layout(
    title='Recommended Total Price: Persona × Temperature',
    xaxis_title='Temperature',
    yaxis_title='Persona',
    height=350,
    template='plotly_white',
)
fig_heat.show()

print('\nInterpretation: higher prices in warmer weather reflect Frappuccino recommendations')
print('with more customization opportunities (Cold Foam, Caramel Drizzle, etc.)')

## Limitations

1. **Content-based only** — no collaborative filtering (intentional: avoids circular reasoning with synthetic data)
2. **Scoring weights are hand-tuned** — with real POS data, these could be learned via click-through rate optimization
3. **Customization suggestions are rule-based** — a production system would use A/B testing to optimize
4. **No flavor/taste modeling** — "Caramel" and "Mocha" are treated equally; real preference varies by individual
5. **Static menu** — doesn't account for out-of-stock items or time-limited offerings

## Value proposition

This notebook demonstrates:
- A **working recommendation pipeline** that takes real-time context (weather, time) as input
- **Transparent scoring logic** where every recommendation can be explained
- **Quantified upsell potential** from personalized customization suggestions
- A **framework** that could be plugged into real POS data with minimal modification